# Benchmark VQE — énergie fondamentale $E_0$ (LiH, BeH₂, H₃⁺)

Ce notebook compare plusieurs stratégies VQE sur **la même base** (même hamiltonien, même espace actif, même optimiseur) pour pouvoir choisir — et améliorer — la meilleure.

**Méthodes comparées**

| Nom | Ansatz | Optimisation |
|---|---|---|
| `UCCSD (fermionique)` | toutes les excitations simples + doubles (Jordan-Wigner) | globale |
| `UCCSD (QEB)` | idem mais en *qubit excitations* (rotations de Givens, ~3× moins de CNOT) | globale |
| `HEA (L=…)` | Hardware-Efficient : couches RY-RZ + échelle de CNOT | globale (3 initialisations aléatoires) |
| `ADAPT (QEB / fermionique)` | ADAPT-VQE, 1 opérateur / itération | réoptimisation complète (warm start) |
| `k-ADAPT (k=2,3)` | k opérateurs / itération | réoptimisation complète |
| `ADAPT séquentiel` | 1 opérateur / itération | seul le nouveau paramètre est optimisé (anciens gelés) |
| `… + réopt finale` | idem | + une optimisation globale à la fin |
| `Greedy-VQE (M=8)` | parmi les M meilleurs gradients, on garde l'opérateur qui fait **réellement** le plus baisser l'énergie | réoptimisation complète |

**Métriques** (pour chaque molécule × méthode)
- `erreur_mHa` = $|E_{VQE} - E_{FCI}|$ en milli-Hartree — **précision chimique : 1,6 mHa**
- `n_cnot`, `n_portes`, `profondeur` : coût du circuit (après décomposition en portes 1 qubit + CNOT)
- `n_params` : nombre de paramètres variationnels
- `n_circuits_hw` : **nombre de circuits qu'il faudrait exécuter sur un vrai QPU** (1 par énergie, 2 par paramètre par gradient — règle du *parameter shift*, + le criblage du pool pour ADAPT). C'est la bonne mesure du « temps quantique ».
- `temps_s` : temps de simulation sur la machine (dépend du PC, à prendre comme indication)

Les résultats sont **ajoutés** à `results/benchmark_results.csv` à chaque exécution (avec une étiquette `RUN_TAG`), pour comparer vos améliorations successives.

> 💡 Sur Colab : *Exécution → Tout exécuter*. Comptez ~1 min pour H₃⁺, quelques minutes pour LiH, ~10-20 min pour BeH₂ avec toutes les méthodes.

## 0. Installation

In [ ]:
# Installe PennyLane si nécessaire (Colab, Aqora, ou local).
import sys, subprocess, importlib.util
if importlib.util.find_spec("pennylane") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pennylane", "pandas", "matplotlib"], check=True)

# (Optionnel, Linux/Colab uniquement) PySCF pour utiliser QCHEM_METHOD = "pyscf" (plus rapide sur grosses molécules).
# subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyscf"], check=True)

# (Optionnel, Colab) Sauvegarder les résultats sur Google Drive pour ne pas les perdre à la déconnexion :
# from google.colab import drive; drive.mount("/content/drive")
# RESULTS_DIR_OVERRIDE = "/content/drive/MyDrive/hackathon_vqe"

In [ ]:
import os, time, json, warnings
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp
from scipy.optimize import minimize
from scipy.sparse.linalg import eigsh

warnings.filterwarnings("ignore")
print("PennyLane", qml.__version__)

## 1. Configuration

Modifiez ici les molécules, l'espace actif, les méthodes à lancer et l'étiquette du run.

**Espace actif** (`active_electrons`, `active_orbitals`) : c'est le levier n°1 pour réduire qubits et portes. Nombre de qubits = 2 × `active_orbitals`.
- H₃⁺ (2 e⁻, 3 orbitales en STO-3G) → 6 qubits, pas besoin de réduire.
- LiH : cœur 1s du Li gelé → 2 e⁻ dans 5 orbitales → 10 qubits.
- BeH₂ : cœur 1s du Be gelé → 4 e⁻ dans 6 orbitales → 12 qubits.

In [ ]:
RUN_TAG = "baseline"            # changez-le à chaque essai (ex. "eps=1e-4", "k=4", ...) pour comparer les runs
BASIS = "sto-3g"
QCHEM_METHOD = "dhf"            # "dhf" (pur Python, marche partout) ou "pyscf" (Linux/Colab, nécessite pyscf)
MAPPING = "jordan_wigner"       # "jordan_wigner", "parity" ou "bravyi_kitaev"
DEVICE_NAME = "lightning.qubit" # simulateur rapide (C++) ; "default.qubit" en secours
CHEM_ACC = 1.6e-3               # précision chimique (Ha)
RESULTS_DIR = globals().get("RESULTS_DIR_OVERRIDE", "results")

# Paramètres ADAPT
ADAPT_EPS = 1e-3                # arrêt quand ||gradient du pool|| < eps
ADAPT_MAX_OPS = 40              # nombre max d'opérateurs dans l'ansatz
GREEDY_M = 8                    # Greedy-VQE : nb de candidats (meilleurs gradients) testés par itération
HEA_RESTARTS = 3                # HEA : nb d'initialisations aléatoires (on garde la meilleure)
SEED = 0

# Géométries (Angström) proches de l'équilibre
MOLECULES = {
    "H3+": dict(symbols=["H", "H", "H"],
                coords=[[0.0, 0.0, 0.0], [0.874, 0.0, 0.0], [0.437, 0.7569, 0.0]],  # triangle équilatéral
                charge=1, mult=1, active_electrons=None, active_orbitals=None),
    "LiH": dict(symbols=["Li", "H"],
                coords=[[0.0, 0.0, 0.0], [0.0, 0.0, 1.5949]],
                charge=0, mult=1, active_electrons=2, active_orbitals=5),
    "BeH2": dict(symbols=["Be", "H", "H"],
                 coords=[[0.0, 0.0, 0.0], [0.0, 0.0, 1.326], [0.0, 0.0, -1.326]],
                 charge=0, mult=1, active_electrons=4, active_orbitals=6),
    # Ajoutez ici la molécule donnée le jour J, ex. :
    # "H2O": dict(symbols=["O","H","H"], coords=[[0,0,0],[0.757,0.586,0],[-0.757,0.586,0]],
    #             charge=0, mult=1, active_electrons=4, active_orbitals=4),
}

MOLECULES_TO_RUN = ["H3+", "LiH", "BeH2"]

METHODS_TO_RUN = [
    "UCCSD (fermionique)",
    "UCCSD (QEB)",
    "HEA (L=2)",
    "HEA (L=4)",
    "ADAPT (fermionique)",
    "ADAPT (QEB)",
    "k-ADAPT (QEB, k=2)",
    "k-ADAPT (QEB, k=3)",
    "ADAPT séquentiel (QEB)",
    "ADAPT séquentiel + réopt finale (QEB)",
    "k-ADAPT séquentiel + réopt finale (QEB, k=2)",
    "Greedy-VQE (QEB)",
]

## 2. Construction du problème : hamiltonien, état Hartree-Fock, énergie exacte (FCI)

L'énergie exacte est obtenue par diagonalisation du hamiltonien **restreinte au bon nombre d'électrons et à $S_z=0$** (sinon, pour un cation comme H₃⁺, le minimum global sur tout l'espace de Fock correspondrait à une autre charge !).

In [ ]:
ANGSTROM_TO_BOHR = 1.8897259886

def build_problem(name, cfg):
    t0 = time.time()
    coords = np.array(cfg["coords"], dtype=float) * ANGSTROM_TO_BOHR
    mol = qml.qchem.Molecule(cfg["symbols"], coords, charge=cfg["charge"],
                             mult=cfg["mult"], basis_name=BASIS)
    n_el = cfg["active_electrons"] or mol.n_electrons
    H, n_qubits = qml.qchem.molecular_hamiltonian(
        mol, active_electrons=cfg["active_electrons"], active_orbitals=cfg["active_orbitals"],
        method=QCHEM_METHOD, mapping=MAPPING)
    wires = list(range(n_qubits))
    hf = qml.qchem.hf_state(n_el, n_qubits)
    if MAPPING != "jordan_wigner":
        hf = qml.qchem.hf_state(n_el, n_qubits, basis=MAPPING)

    # --- Énergie exacte dans le secteur (N électrons, Sz = (mult-1)/2) ---
    Hs = H.sparse_matrix(wire_order=wires).tocsr()
    idx = np.arange(2 ** n_qubits)
    bits = (idx[:, None] >> (n_qubits - 1 - np.arange(n_qubits))[None, :]) & 1
    if MAPPING == "jordan_wigner":
        n_occ = bits.sum(1)
        two_sz = bits[:, 0::2].sum(1) - bits[:, 1::2].sum(1)
        sel = np.where((n_occ == n_el) & (two_sz == cfg["mult"] - 1))[0]
    else:  # pour parity/BK l'occupation n'est pas lisible directement : diagonalisation complète
        sel = idx
    Hsub = Hs[sel][:, sel]
    if len(sel) <= 3000:
        e_fci = np.linalg.eigvalsh(Hsub.toarray()).min()
    else:
        e_fci = eigsh(Hsub, k=1, which="SA")[0][0]
    hf_index = int("".join(map(str, hf)), 2)
    e_hf = Hs[hf_index, hf_index].real

    singles, doubles = qml.qchem.excitations(n_el, n_qubits)
    prob = dict(name=name, H=H, n_qubits=n_qubits, n_electrons=n_el, wires=wires,
                hf_state=np.array(hf), E_fci=float(e_fci), E_hf=float(e_hf),
                singles=singles, doubles=doubles, n_terms=len(H.operands) if hasattr(H, "operands") else None)
    print(f"{name:6s} | {n_qubits:2d} qubits | {n_el} e- | {len(singles)} simples, {len(doubles)} doubles | "
          f"E_HF = {e_hf:.6f} | E_FCI = {e_fci:.6f} Ha | corrélation = {1e3*(e_hf-e_fci):.2f} mHa "
          f"| {time.time()-t0:.1f}s")
    return prob

## 3. Briques : ansatz, simulateur, optimiseur, comptage de portes

Un ansatz est représenté par une **liste d'opérateurs** `(type, wires)` + un vecteur de paramètres :
- `S`, `D` : excitations **qubit** simple/double (`qml.SingleExcitation` / `qml.DoubleExcitation`, pool *QEB*)
- `FS`, `FD` : excitations **fermioniques** (`qml.FermionicSingleExcitation` / `qml.FermionicDoubleExcitation`, UCCSD standard)
- `RY`, `RZ`, `CNOT` : portes de l'ansatz Hardware-Efficient

In [ ]:
PARAM_KINDS = {"S", "D", "FS", "FD", "RY", "RZ"}

def apply_op(kind, w, theta=None):
    if kind == "S":    qml.SingleExcitation(theta, wires=list(w))
    elif kind == "D":  qml.DoubleExcitation(theta, wires=list(w))
    elif kind == "FS": qml.FermionicSingleExcitation(theta, wires=list(w))
    elif kind == "FD": qml.FermionicDoubleExcitation(theta, wires1=list(w[0]), wires2=list(w[1]))
    elif kind == "RY": qml.RY(theta, wires=w[0])
    elif kind == "RZ": qml.RZ(theta, wires=w[0])
    elif kind == "CNOT": qml.CNOT(wires=list(w))
    else: raise ValueError(kind)

def n_params_of(ops):
    return sum(k in PARAM_KINDS for k, _ in ops)

def apply_ansatz(ops, params):
    i = 0
    for kind, w in ops:
        if kind in PARAM_KINDS:
            apply_op(kind, w, params[i]); i += 1
        else:
            apply_op(kind, w)

# ---- Pools d'opérateurs ----
def qeb_pool(prob):
    return [("D", tuple(d)) for d in prob["doubles"]] + [("S", tuple(s)) for s in prob["singles"]]

def fermionic_pool(prob):
    s_w, d_w = qml.qchem.excitations_to_wires(prob["singles"], prob["doubles"])
    return [("FD", (tuple(w1), tuple(w2))) for w1, w2 in d_w] + [("FS", tuple(w)) for w in s_w]

def hea_ops(n_qubits, layers):
    ops = []
    for _ in range(layers):
        for q in range(n_qubits):
            ops += [("RY", (q,)), ("RZ", (q,))]
        for q in range(n_qubits - 1):
            ops.append(("CNOT", (q, q + 1)))
    return ops

In [ ]:
class Estimator:
    # Simule E(theta) = <HF| U(theta)^dag H U(theta) |HF> et son gradient (méthode adjoint).
    # Compte aussi le coût "matériel" équivalent (nb de circuits à exécuter sur un vrai QPU).
    def __init__(self, prob):
        self.prob = prob
        try:
            self.dev = qml.device(DEVICE_NAME, wires=prob["n_qubits"])
        except Exception:
            self.dev = qml.device("default.qubit", wires=prob["n_qubits"])
        self._cache = {}
        self.n_energy = 0      # nb d'évaluations d'énergie
        self.n_grad = 0        # nb d'évaluations de gradient
        self.n_hw = 0          # circuits équivalents QPU (parameter-shift : 2 par paramètre)

    def _qnode(self, ops):
        key = tuple(ops)
        if key not in self._cache:
            prob = self.prob
            @qml.qnode(self.dev, diff_method="adjoint")
            def circuit(params):
                qml.BasisState(prob["hf_state"], wires=prob["wires"])
                apply_ansatz(ops, params)
                return qml.expval(prob["H"])
            self._cache = {key: circuit}   # on ne garde que le dernier (économie mémoire)
        return self._cache[key]

    def energy(self, ops, x):
        self.n_energy += 1; self.n_hw += 1
        if len(x) == 0:
            return self.prob["E_hf"]
        return float(self._qnode(ops)(pnp.array(x, requires_grad=True)))

    def grad(self, ops, x, n_hw_params=None):
        self.n_grad += 1
        self.n_hw += 2 * (len(x) if n_hw_params is None else n_hw_params)
        g = qml.grad(self._qnode(ops))(pnp.array(x, requires_grad=True))
        return np.asarray(g, dtype=float)


def optimize(est, ops, x0, free=None, maxiter=500):
    # BFGS sur les paramètres `free` (les autres restent gelés).
    x0 = np.array(x0, dtype=float)
    if len(x0) == 0:
        return x0, est.energy(ops, x0)
    free = np.arange(len(x0)) if free is None else np.array(free)
    def full(y):
        x = x0.copy(); x[free] = y; return x
    f = lambda y: est.energy(ops, full(y))
    g = lambda y: est.grad(ops, full(y), n_hw_params=len(free))[free]
    res = minimize(f, x0[free], jac=g, method="BFGS", options=dict(gtol=1e-6, maxiter=maxiter))
    return full(res.x), float(res.fun)

In [ ]:
def _decompose(op):
    # Décompose récursivement jusqu'aux portes 1 qubit + CNOT/CZ.
    if len(op.wires) == 1 or op.name in ("CNOT", "CZ"):
        return [op]
    try:
        dec = op.decomposition()
    except Exception:
        return [op]
    out = []
    for o in dec:
        out += _decompose(o)
    return out

def circuit_resources(ops):
    # Compte CNOT, portes totales et profondeur de l'ansatz (hors préparation HF = quelques X).
    x = np.full(n_params_of(ops), 0.1)
    with qml.queuing.AnnotatedQueue() as q:
        apply_ansatz(ops, x)
    tape = qml.tape.QuantumScript.from_queue(q)
    gates = [g for op in tape.operations for g in _decompose(op)
             if g.name not in ("GlobalPhase", "Identity")]
    layer = {}
    for g in gates:
        d = max(layer.get(w, 0) for w in g.wires) + 1
        for w in g.wires:
            layer[w] = d
    return dict(n_cnot=sum(len(g.wires) >= 2 for g in gates), n_portes=len(gates),
                profondeur=max(layer.values()) if layer else 0, n_params=len(x))

## 4. Les méthodes

### Ansatz fixes (UCCSD, HEA)
On construit tout l'ansatz d'un coup et on optimise tous les paramètres.

### Famille ADAPT
À chaque itération : (1) on calcule le gradient $\partial E/\partial\theta$ de **chaque opérateur du pool** ajouté en fin de circuit avec $\theta=0$ (une seule passe adjoint suffit) ; (2) on ajoute les `k` opérateurs de plus fort |gradient| (ou, pour Greedy, celui qui baisse **réellement** le plus l'énergie) ; (3) on optimise — soit **tout** (warm start), soit **seulement les nouveaux** paramètres (mode séquentiel / gelé) ; (4) arrêt quand $\|g\| <$ `ADAPT_EPS`.

In [ ]:
def run_fixed(est, prob, ops, x0=None):
    x0 = np.zeros(n_params_of(ops)) if x0 is None else x0
    x, E = optimize(est, ops, x0)
    return dict(ops=ops, x=x, E=E, hist=[])

def run_hea(est, prob, layers, restarts=HEA_RESTARTS):
    ops = hea_ops(prob["n_qubits"], layers)
    rng = np.random.default_rng(SEED)
    best = None
    for _ in range(restarts):
        x0 = rng.normal(0, 0.1, n_params_of(ops))
        x, E = optimize(est, ops, x0)
        if best is None or E < best["E"]:
            best = dict(ops=ops, x=x, E=E, hist=[])
    return best

def run_adapt(est, prob, pool, k=1, sequential=False, final_reopt=False, greedy_M=None,
              eps=None, max_ops=None):
    eps = ADAPT_EPS if eps is None else eps
    max_ops = ADAPT_MAX_OPS if max_ops is None else max_ops
    ops, x, E = [], np.zeros(0), prob["E_hf"]
    hist = [dict(n_ops=0, E=E, gnorm=np.nan, n_hw=est.n_hw)]
    while len(ops) < max_ops:
        # 1) gradients de tout le pool, en une passe
        ext = list(ops) + pool
        g = est.grad(ext, np.concatenate([x, np.zeros(len(pool))]), n_hw_params=len(pool))[len(x):]
        gnorm = float(np.linalg.norm(g))
        if gnorm < eps:
            break
        # 2) choix des opérateurs
        if greedy_M:
            cand = np.argsort(-np.abs(g))[:greedy_M]
            trial_E = []
            for c in cand:
                _, Ec = optimize(est, ops + [pool[c]], np.append(x, 0.0), free=[len(x)])
                trial_E.append(Ec)
            chosen = list(cand[np.argsort(trial_E)][:k])
        else:
            chosen = list(np.argsort(-np.abs(g))[:k])
        ops = ops + [pool[c] for c in chosen]
        x = np.append(x, np.zeros(len(chosen)))
        # 3) optimisation
        free = list(range(len(x) - len(chosen), len(x))) if sequential else None
        x, E_new = optimize(est, ops, x, free=free)
        hist.append(dict(n_ops=len(ops), E=E_new, gnorm=gnorm, n_hw=est.n_hw))
        if abs(E - E_new) < 1e-9:   # plus aucun progrès
            E = E_new; break
        E = E_new
    if final_reopt and len(x):
        x, E = optimize(est, ops, x)
        hist.append(dict(n_ops=len(ops), E=E, gnorm=np.nan, n_hw=est.n_hw, final=True))
    return dict(ops=ops, x=x, E=E, hist=hist)


METHODS = {
    "UCCSD (fermionique)": lambda est, p: run_fixed(est, p, fermionic_pool(p)),
    "UCCSD (QEB)":         lambda est, p: run_fixed(est, p, qeb_pool(p)),
    "HEA (L=2)":           lambda est, p: run_hea(est, p, 2),
    "HEA (L=4)":           lambda est, p: run_hea(est, p, 4),
    "ADAPT (fermionique)": lambda est, p: run_adapt(est, p, fermionic_pool(p)),
    "ADAPT (QEB)":         lambda est, p: run_adapt(est, p, qeb_pool(p)),
    "k-ADAPT (QEB, k=2)":  lambda est, p: run_adapt(est, p, qeb_pool(p), k=2),
    "k-ADAPT (QEB, k=3)":  lambda est, p: run_adapt(est, p, qeb_pool(p), k=3),
    "ADAPT séquentiel (QEB)":
        lambda est, p: run_adapt(est, p, qeb_pool(p), sequential=True),
    "ADAPT séquentiel + réopt finale (QEB)":
        lambda est, p: run_adapt(est, p, qeb_pool(p), sequential=True, final_reopt=True),
    "k-ADAPT séquentiel + réopt finale (QEB, k=2)":
        lambda est, p: run_adapt(est, p, qeb_pool(p), k=2, sequential=True, final_reopt=True),
    "Greedy-VQE (QEB)":
        lambda est, p: run_adapt(est, p, qeb_pool(p), greedy_M=GREEDY_M),
}

## 5. Lancement du benchmark

In [ ]:
def run_benchmark(mol_names, method_names):
    rows, histories, problems = [], {}, {}
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    for mname in mol_names:
        prob = build_problem(mname, MOLECULES[mname])
        problems[mname] = prob
        for meth in method_names:
            est = Estimator(prob)
            t0 = time.time()
            try:
                out = METHODS[meth](est, prob)
            except Exception as e:
                print(f"   ✗ {meth}: {type(e).__name__}: {e}")
                continue
            dt = time.time() - t0
            res = circuit_resources(out["ops"])
            err = abs(out["E"] - prob["E_fci"])
            rows.append(dict(run_tag=RUN_TAG, date=stamp, molecule=mname, methode=meth,
                             n_qubits=prob["n_qubits"], E=out["E"], E_fci=prob["E_fci"],
                             erreur_mHa=1e3 * err, precision_chimique=err < CHEM_ACC,
                             **res, n_circuits_hw=est.n_hw, n_eval_E=est.n_energy,
                             n_eval_grad=est.n_grad, temps_s=dt))
            histories[(mname, meth)] = out["hist"]
            flag = "✓" if err < CHEM_ACC else "✗"
            print(f"   {flag} {meth:46s} err = {1e3*err:9.4f} mHa | CNOT = {res['n_cnot']:5d} | "
                  f"params = {res['n_params']:3d} | circuits QPU ≈ {est.n_hw:8d} | {dt:6.1f}s")
    return pd.DataFrame(rows), histories, problems

df, histories, problems = run_benchmark(MOLECULES_TO_RUN, METHODS_TO_RUN)

os.makedirs(RESULTS_DIR, exist_ok=True)
csv_path = os.path.join(RESULTS_DIR, "benchmark_results.csv")
df.to_csv(csv_path, mode="a", header=not os.path.exists(csv_path), index=False)
print(f"\nRésultats ajoutés à {csv_path}")

## 6. Classement

Pour chaque molécule, les méthodes sont classées par : **(1)** précision chimique atteinte, **(2)** nombre de CNOT croissant, **(3)** nombre de circuits QPU croissant.

In [ ]:
cols = ["methode", "erreur_mHa", "precision_chimique", "n_cnot", "profondeur", "n_params", "n_circuits_hw", "temps_s"]
for mname, d in df.groupby("molecule", sort=False):
    d = d.sort_values(["precision_chimique", "n_cnot", "n_circuits_hw"], ascending=[False, True, True])
    print(f"\n=== {mname} ({d.n_qubits.iloc[0]} qubits) ===")
    t = d[cols].reset_index(drop=True)
    try:   # tableau coloré (nécessite jinja2, présent sur Colab)
        display(t.style.format({"erreur_mHa": "{:.4f}", "temps_s": "{:.1f}"})
                .background_gradient(subset=["erreur_mHa"], cmap="RdYlGn_r")
                .background_gradient(subset=["n_cnot"], cmap="RdYlGn_r"))
    except Exception:
        display(t.round(4))

## 7. Visualisations

### 7.1 Précision vs coût en portes (front de Pareto)
**Le meilleur modèle est en bas à gauche** : sous la ligne de précision chimique, avec le moins de CNOT.

In [ ]:
METH_ORDER = [m for m in METHODS_TO_RUN if m in df.methode.unique()]
cmap = plt.get_cmap("tab20")
COLORS = {m: cmap(i % 20) for i, m in enumerate(METH_ORDER)}
MARKERS = {m: "osD^v<>phX*P"[i % 12] for i, m in enumerate(METH_ORDER)}
mols = list(df.molecule.unique())

def _floor(e): return np.maximum(e, 1e-7)   # pour l'échelle log

fig, axes = plt.subplots(1, len(mols), figsize=(6 * len(mols), 5), squeeze=False)
for ax, mname in zip(axes[0], mols):
    d = df[df.molecule == mname]
    for _, r in d.iterrows():
        ax.scatter(max(r.n_cnot, 1), _floor(r.erreur_mHa), s=90, color=COLORS[r.methode],
                   marker=MARKERS[r.methode], edgecolor="k", label=r.methode, zorder=3)
    ax.axhline(1e3 * CHEM_ACC, color="green", ls="--", lw=1.5, label="précision chimique")
    ax.axhspan(1e-7, 1e3 * CHEM_ACC, color="green", alpha=0.07)
    ax.set(xscale="log", yscale="log", xlabel="Nombre de CNOT (log)", ylabel="Erreur |E - E_FCI| (mHa, log)",
           title=f"{mname} — {d.n_qubits.iloc[0]} qubits")
    ax.grid(alpha=0.3, which="both")
h, l = axes[0][0].get_legend_handles_labels()
uniq = dict(zip(l, h))
fig.legend(uniq.values(), uniq.keys(), loc="lower center", ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.12))
plt.tight_layout(); plt.show()

### 7.2 Erreur et coût par méthode

In [ ]:
metrics = [("erreur_mHa", "Erreur (mHa, log)", True), ("n_cnot", "CNOT", False),
           ("n_circuits_hw", "Circuits QPU équivalents (log)", True)]
fig, axes = plt.subplots(len(mols), 3, figsize=(18, 0.45 * len(METH_ORDER) * len(mols) + 1.5), squeeze=False)
for i, mname in enumerate(mols):
    d = df[df.molecule == mname].set_index("methode").reindex(METH_ORDER).dropna(subset=["E"])
    for j, (col, lab, log) in enumerate(metrics):
        ax = axes[i][j]
        vals = d[col].astype(float).values
        if col == "erreur_mHa":
            colors = ["#2ca02c" if ok else "#d62728" for ok in d.precision_chimique]
            vals = _floor(vals)
        else:
            colors = [COLORS[m] for m in d.index]
        ax.barh(d.index, vals, color=colors, edgecolor="k")
        if col == "erreur_mHa":
            ax.axvline(1e3 * CHEM_ACC, color="green", ls="--")
        if log: ax.set_xscale("log")
        ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)
        ax.set_title(f"{mname} — {lab}")
        if j > 0: ax.set_yticklabels([])
plt.tight_layout(); plt.show()

### 7.3 Convergence des méthodes adaptatives
À gauche : erreur en fonction du nombre d'opérateurs (≈ taille du circuit). À droite : erreur en fonction du nombre de circuits QPU dépensés (≈ temps quantique).

In [ ]:
adaptive = [m for m in METH_ORDER if any(len(histories.get((mn, m), [])) for mn in mols)]
fig, axes = plt.subplots(len(mols), 2, figsize=(15, 4.5 * len(mols)), squeeze=False)
for i, mname in enumerate(mols):
    Ef = problems[mname]["E_fci"]
    for m in adaptive:
        h = histories.get((mname, m), [])
        if not h: continue
        err = _floor(1e3 * np.abs(np.array([e["E"] for e in h]) - Ef))
        nops = [e["n_ops"] for e in h]; nhw = [max(e["n_hw"], 1) for e in h]
        axes[i][0].plot(nops, err, marker=MARKERS[m], color=COLORS[m], label=m, ms=4)
        axes[i][1].plot(nhw, err, marker=MARKERS[m], color=COLORS[m], label=m, ms=4)
    for j, xl in enumerate(["Nombre d'opérateurs dans l'ansatz", "Circuits QPU équivalents (log)"]):
        ax = axes[i][j]
        ax.axhline(1e3 * CHEM_ACC, color="green", ls="--")
        ax.set(yscale="log", xlabel=xl, ylabel="Erreur (mHa)", title=f"{mname}")
        if j == 1: ax.set_xscale("log")
        ax.grid(alpha=0.3, which="both")
    axes[i][0].legend(fontsize=8)
plt.tight_layout(); plt.show()

### 7.4 Vue d'ensemble : carte méthodes × molécules
Couleur = erreur (vert = précision chimique) ; texte = erreur en mHa / nombre de CNOT.

In [ ]:
piv_e = df.pivot_table(index="methode", columns="molecule", values="erreur_mHa").reindex(METH_ORDER)[mols]
piv_c = df.pivot_table(index="methode", columns="molecule", values="n_cnot").reindex(METH_ORDER)[mols]
fig, ax = plt.subplots(figsize=(2.6 * len(mols) + 4, 0.55 * len(METH_ORDER) + 1.5))
im = ax.imshow(np.log10(_floor(piv_e.values.astype(float))), cmap="RdYlGn_r", aspect="auto",
               vmin=-4, vmax=2)
for a in range(piv_e.shape[0]):
    for b in range(piv_e.shape[1]):
        e, c = piv_e.values[a, b], piv_c.values[a, b]
        if np.isfinite(e):
            ax.text(b, a, f"{e:.3g} mHa\n{int(c)} CNOT", ha="center", va="center", fontsize=8)
ax.set_xticks(range(len(mols)), mols); ax.set_yticks(range(len(METH_ORDER)), METH_ORDER)
cb = plt.colorbar(im, ax=ax); cb.set_label("log10(erreur en mHa)  —  précision chimique = 0.2")
ax.set_title("Erreur / CNOT par méthode et molécule")
plt.tight_layout(); plt.show()

## 8. Historique : comparer avec les runs précédents

Chaque exécution ajoute ses lignes à `benchmark_results.csv` avec son `RUN_TAG`. Changez un réglage (ex. `ADAPT_EPS`, `k`, espace actif, mapping, nouvelle méthode…), mettez un nouveau `RUN_TAG`, relancez, puis regardez ici si ça s'améliore.

In [ ]:
hist_df = pd.read_csv(csv_path)
print(f"{len(hist_df)} lignes, runs : {list(hist_df.run_tag.unique())}")
for metric in ["erreur_mHa", "n_cnot", "n_circuits_hw"]:
    print(f"\n--- {metric} (dernier résultat de chaque run) ---")
    display(hist_df.groupby(["molecule", "methode", "run_tag"], sort=False)[metric].last()
            .unstack("run_tag").round(4))

In [ ]:
# Meilleur compromis global trouvé jusqu'ici, par molécule (précision chimique puis moins de CNOT)
best = (hist_df[hist_df.precision_chimique]
        .sort_values(["n_cnot", "n_circuits_hw"])
        .groupby("molecule").head(3)
        [["molecule", "run_tag", "methode", "erreur_mHa", "n_cnot", "n_params", "n_circuits_hw"]])
display(best.sort_values(["molecule", "n_cnot"]).reset_index(drop=True))

## 9. Pistes d'amélioration à tester (une à la fois, avec un nouveau `RUN_TAG`)

1. **`ADAPT_EPS`** : plus grand (1e-2) → circuits plus courts ; plus petit → plus précis mais plus de portes. ⚠️ Sur BeH₂, ADAPT traverse une « vallée » de gradient quasi nul (‖g‖ ≈ 4e-5 à 18 opérateurs, erreur 0,37 mHa) puis repart : il faut `ADAPT_EPS = 1e-5` pour descendre à ~0,0003 mHa (35 opérateurs). Le mode séquentiel + réopt finale échappe naturellement à ce piège.
2. **`k`** dans k-ADAPT (2, 3, 4…) → compromis nombre d'itérations / taille du circuit.
3. **Réoptimisation périodique** dans le mode séquentiel (tous les N opérateurs) : à ajouter dans `run_adapt`.
4. **Espace actif** plus petit (ex. LiH 2e/3o = 6 qubits) : énorme gain de portes, mais $E_{FCI}$ de l'espace actif s'éloigne de la vraie énergie — à vérifier avec ce que le jury attend.
5. **Mapping** `parity` / `bravyi_kitaev` + réduction de qubits par symétries (`qml.qchem.taper`).
6. **Autres pools** : qubit-ADAPT (chaînes de Pauli), pools restreints aux excitations respectant la symétrie spatiale.
7. **Optimiseur** : L-BFGS-B, ou `qml.RotosolveOptimizer` (très efficace en mode séquentiel).

Pour ajouter une méthode : écrivez une fonction `(est, prob) -> dict(ops, x, E, hist)` et ajoutez-la dans `METHODS` et `METHODS_TO_RUN`.